# Trivima — Real Multi-View Training (CLIP + Real Poses)

**The right approach combined into one notebook:**
- CLIP ViT-L image encoder (strong scene conditioning)
- Real RE10K pixelSplat .torch data (multiple frames per scene + real camera poses)
- Relative pose conditioning (Zero123/SeVA style)
- Single-target training: `(input_photo, relative_pose) → target_view`
- Drive auto-resume
- Optimized data pipeline

**Inference**: call the model 8 times with 8 poses → 8 multi-view images

## 1. Install + mount Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q diffusers transformers peft accelerate gdown huggingface_hub

from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/trivima_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted: {DRIVE_DIR}')
!ls -lh {DRIVE_DIR}/ 2>/dev/null || echo 'no existing checkpoints'

## 2. Download pixelSplat RE10K subset (real multi-view, ~600MB)

In [ ]:
import gdown, zipfile, glob

os.makedirs('/content/data', exist_ok=True)

TORCH_DIR = '/content/data/re10k_subset'
torch_files = glob.glob(f'{TORCH_DIR}/**/*.torch', recursive=True)

if len(torch_files) == 0:
    print('Downloading pixelSplat RE10K subset from gdrive...')
    gdown.download_folder(
        'https://drive.google.com/drive/folders/1joiezNCyQK2BvWMnfwHJpm2V77c7iYGe',
        output='/content/data/re10k_raw',
    )
    print('Download done. Inspecting...')
    !ls -la /content/data/re10k_raw/

    # Extract any zips found
    zips = glob.glob('/content/data/re10k_raw/**/*.zip', recursive=True)
    print(f'Found {len(zips)} zip files')
    for z in zips:
        print(f'Extracting {z}...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(TORCH_DIR)

    # If no zips, the gdrive folder may already contain .torch files directly
    if not zips:
        print('No zips — checking if .torch files are direct...')
        direct_torches = glob.glob('/content/data/re10k_raw/**/*.torch', recursive=True)
        if direct_torches:
            os.makedirs(TORCH_DIR, exist_ok=True)
            for t in direct_torches:
                import shutil
                shutil.move(t, os.path.join(TORCH_DIR, os.path.basename(t)))
            print(f'Moved {len(direct_torches)} .torch files')

    torch_files = glob.glob(f'{TORCH_DIR}/**/*.torch', recursive=True)

print(f'\n.torch files available: {len(torch_files)}')
if len(torch_files) == 0:
    raise RuntimeError('No .torch files found. Inspect /content/data/ manually.')
for f in torch_files[:5]:
    sz = os.path.getsize(f) / 1e6
    print(f'  {f} ({sz:.1f} MB)')

## 3. Load SD 1.5 + LoRA + CLIP

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPVisionModel

device = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(device)
vae.requires_grad_(False); vae.eval()

clip_vision = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_vision.requires_grad_(False); clip_vision.eval()
CLIP_DIM = 1024

unet = UNet2DConditionModel.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', subfolder='unet'
).to(device)
try:
    from diffusers.models.attention_processor import AttnProcessor2_0
    unet.set_attn_processor(AttnProcessor2_0())
    print('SDPA flash attention enabled')
except Exception as e:
    print(f'SDPA warning: {e}')

# PRODUCTION: r=32 (4x more capacity than r=16)
lora_config = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.1,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0'],
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# CLIP 1024 -> SD 768 (257 image tokens)
clip_proj = nn.Sequential(
    nn.LayerNorm(CLIP_DIM),
    nn.Linear(CLIP_DIM, 768),
).to(device)

# Relative pose -> 768 (4 tokens for richer signal)
class RelPoseEncoder(nn.Module):
    def __init__(self, dim=768, n_tokens=4):
        super().__init__()
        self.n_tokens = n_tokens
        self.mlp = nn.Sequential(
            nn.Linear(16, dim*2), nn.SiLU(),
            nn.Linear(dim*2, dim*n_tokens),
        )
    def forward(self, pose):
        B = pose.shape[0]
        return self.mlp(pose.reshape(B, -1)).reshape(B, self.n_tokens, -1)

pose_encoder = RelPoseEncoder(768, n_tokens=4).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.alpha_cumprod = scheduler.alphas_cumprod.to(device)
print('Models ready (PRODUCTION: LoRA r=32)')

## 4. Real multi-view dataset (samples 2 frames per scene with relative pose)

In [ ]:
import io, glob, random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

class RE10KMultiView(Dataset):
    """Loads pixelSplat .torch files. Each item samples a random (input, target) frame pair from a scene."""
    def __init__(self, torch_glob, sd_size=256, max_scenes=None, max_frame_gap=20, samples_per_scene=8):
        self.files = sorted(glob.glob(torch_glob, recursive=True))
        if not self.files:
            raise RuntimeError(f'No .torch files matched: {torch_glob}')
        print(f'Found {len(self.files)} .torch files, loading scenes into RAM...')
        self.scenes = []
        for fp in self.files:
            data = torch.load(fp, map_location='cpu', weights_only=False)
            for scene in data:
                if len(scene['images']) >= 2:
                    self.scenes.append(scene)
                    if max_scenes and len(self.scenes) >= max_scenes:
                        break
            if max_scenes and len(self.scenes) >= max_scenes:
                break
        if not self.scenes:
            raise RuntimeError('No usable scenes found (all have < 2 frames)')
        print(f'Loaded {len(self.scenes)} scenes')
        self.samples_per_scene = samples_per_scene
        self.sd_size = sd_size
        self.max_gap = max_frame_gap
        self.sd_tf = T.Compose([
            T.Resize((sd_size, sd_size), T.InterpolationMode.LANCZOS),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])
        self.clip_tf = T.Compose([
            T.Resize((224, 224), T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(CLIP_MEAN, CLIP_STD),
        ])

    def __len__(self):
        return len(self.scenes) * self.samples_per_scene

    def _decode(self, img_bytes):
        return Image.open(io.BytesIO(img_bytes.numpy().tobytes())).convert('RGB')

    def _pose_4x4(self, cam_18):
        ext = cam_18[6:18].reshape(3, 4)
        pose = np.eye(4, dtype=np.float32)
        pose[:3, :4] = ext
        return pose

    def __getitem__(self, idx):
        scene = self.scenes[idx % len(self.scenes)]
        n = len(scene['images'])
        i_in = random.randint(0, n - 1)
        gap = random.randint(1, min(self.max_gap, n - 1))
        i_tg = (i_in + gap) % n

        img_in = self._decode(scene['images'][i_in])
        img_tg = self._decode(scene['images'][i_tg])

        cams = scene['cameras'].numpy()
        pose_in = self._pose_4x4(cams[i_in])
        pose_tg = self._pose_4x4(cams[i_tg])

        try:
            pose_in_inv = np.linalg.inv(pose_in)
        except np.linalg.LinAlgError:
            pose_in_inv = np.eye(4, dtype=np.float32)
        rel = (pose_in_inv @ pose_tg).astype(np.float32)

        return {
            'input_sd':   self.sd_tf(img_in),
            'input_clip': self.clip_tf(img_in),
            'target':     self.sd_tf(img_tg),
            'rel_pose':   torch.from_numpy(rel),
        }

# PRODUCTION: use ALL scenes, more samples per scene
dataset = RE10KMultiView(
    f'{TORCH_DIR}/**/*.torch',
    sd_size=256,
    max_scenes=None,        # use everything
    max_frame_gap=20,
    samples_per_scene=16,   # more pair sampling per epoch
)
loader = DataLoader(
    dataset, batch_size=24, shuffle=True, num_workers=8,
    drop_last=True, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f'Dataset items: {len(dataset)}, batches/epoch: {len(loader)}')

## 5. PRODUCTION Train (5 hours, LoRA r=32, all data, cosine LR + warmup, auto-resume)

In [ ]:
import time, math, numpy as np

# === PRODUCTION TRAINING (5 hours target) ===
EPOCHS = 200                         # large cap; time-budget kicks in first
LR_PEAK = 1.5e-4
LR_MIN = 1e-6
WARMUP_STEPS = 500
TIME_BUDGET_SEC = 5 * 3600           # stop after 5 hours
COND_DROPOUT = 0.1

# New checkpoint paths so old r=16 weights aren't touched
BEST_PATH   = f'{DRIVE_DIR}/best_mv_prod.pt'
RESUME_PATH = f'{DRIVE_DIR}/last_mv_prod.pt'

trainable = list(unet.parameters()) + list(clip_proj.parameters()) + list(pose_encoder.parameters())
optimizer = torch.optim.AdamW(
    [p for p in trainable if p.requires_grad],
    lr=LR_PEAK, weight_decay=0.01, betas=(0.9, 0.999),
)

# Cosine decay with warmup (manual, so resume works correctly)
total_steps_estimate = EPOCHS * len(loader)
def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, total_steps_estimate - WARMUP_STEPS)
    p = min(1.0, p)
    return LR_MIN + 0.5 * (LR_PEAK - LR_MIN) * (1 + math.cos(math.pi * p))

best_loss = float('inf')
start_epoch = 0
global_step = 0

if os.path.exists(RESUME_PATH):
    print(f'Resuming from {RESUME_PATH}')
    ck = torch.load(RESUME_PATH, map_location=device)
    unet.load_state_dict(ck['unet'], strict=False)
    clip_proj.load_state_dict(ck['clip_proj'])
    pose_encoder.load_state_dict(ck['pose_encoder'])
    optimizer.load_state_dict(ck['optimizer'])
    start_epoch = ck['epoch'] + 1
    best_loss = ck.get('best_loss', float('inf'))
    global_step = ck.get('global_step', start_epoch * len(loader))
    print(f'Resumed epoch={start_epoch} step={global_step} best_loss={best_loss:.4f}')
else:
    print('Fresh production run (LoRA r=32, all scenes, 5h budget)')

def save_best(loss, epoch):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'clip_proj': clip_proj.state_dict(),
        'pose_encoder': pose_encoder.state_dict(),
        'loss': loss, 'epoch': epoch,
    }, BEST_PATH)

def save_resume(epoch, step):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'clip_proj': clip_proj.state_dict(),
        'pose_encoder': pose_encoder.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch, 'best_loss': best_loss, 'global_step': step,
    }, RESUME_PATH)

t_start = time.time()
stop_training = False

for epoch in range(start_epoch, EPOCHS):
    if stop_training:
        break
    t0 = time.time()
    losses = []
    unet.train(); clip_proj.train(); pose_encoder.train()

    for bi, batch in enumerate(loader):
        # Time budget check
        elapsed = time.time() - t_start
        if elapsed > TIME_BUDGET_SEC:
            print(f'\nTime budget reached ({elapsed/3600:.2f}h) — stopping')
            stop_training = True
            break

        # LR schedule
        for pg in optimizer.param_groups:
            pg['lr'] = lr_at(global_step)

        inp_sd   = batch['input_sd'].to(device, non_blocking=True)
        inp_clip = batch['input_clip'].to(device, non_blocking=True)
        tgt      = batch['target'].to(device, non_blocking=True)
        rel_pose = batch['rel_pose'].to(device, non_blocking=True)
        B = inp_sd.shape[0]

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            tgt_lat = vae.encode(tgt).latent_dist.sample() * 0.18215
            clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state

        with torch.autocast('cuda', dtype=torch.bfloat16):
            cond_clip = clip_proj(clip_feats)
            cond_pose = pose_encoder(rel_pose)
            enc_h = torch.cat([cond_clip, cond_pose], dim=1)

            if COND_DROPOUT > 0:
                drop = (torch.rand(B, device=device) < COND_DROPOUT).view(B, 1, 1)
                enc_h = torch.where(drop, torch.zeros_like(enc_h), enc_h)

            t = torch.randint(0, 1000, (B,), device=device).long()
            noise = torch.randn_like(tgt_lat)
            noisy = scheduler.add_noise(tgt_lat, noise, t)

            pred = unet(noisy, t, encoder_hidden_states=enc_h).sample
            loss = F.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        losses.append(loss.item())
        global_step += 1

        if (bi+1) % 50 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'  [{epoch+1}] {bi+1}/{len(loader)} loss={np.mean(losses[-50:]):.4f} lr={cur_lr:.2e} elapsed={(time.time()-t_start)/60:.1f}m')

    if not losses:
        break
    avg = float(np.mean(losses))
    dt = time.time() - t0
    elapsed_h = (time.time() - t_start) / 3600
    print(f'Epoch {epoch+1}: loss={avg:.4f} ({dt:.0f}s) total={elapsed_h:.2f}h/{TIME_BUDGET_SEC/3600:.1f}h')

    save_resume(epoch, global_step)
    if avg < best_loss:
        best_loss = avg
        save_best(best_loss, epoch)
        print(f'  -> new best: {best_loss:.4f}')

print(f'\nDone! Best loss: {best_loss:.4f}, total time: {(time.time()-t_start)/3600:.2f}h')

## 6. Generate target view at a specific relative pose (single view)

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(BEST_PATH, map_location=device)
unet.load_state_dict(ckpt['unet'], strict=False)
clip_proj.load_state_dict(ckpt['clip_proj'])
pose_encoder.load_state_dict(ckpt['pose_encoder'])
unet.eval(); clip_proj.eval(); pose_encoder.eval()
print(f"Loaded best, loss={ckpt['loss']:.4f}")

batch = next(iter(loader))
inp_sd   = batch['input_sd'][:1].to(device)
inp_clip = batch['input_clip'][:1].to(device)
gt       = batch['target'][:1].to(device)
rel_pose = batch['rel_pose'][:1].to(device)

GUIDANCE = 5.0
STRENGTH = 0.7

def generate(inp_sd, inp_clip, rel_pose, guidance=5.0, strength=0.7, n_steps=50):
    with torch.no_grad():
        inp_lat = vae.encode(inp_sd).latent_dist.sample() * 0.18215
        clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state
        cond_clip = clip_proj(clip_feats)
        cond_pose = pose_encoder(rel_pose)
        enc_cond = torch.cat([cond_clip, cond_pose], dim=1)
        enc_uncond = torch.zeros_like(enc_cond)

        start_step = int(n_steps * strength)
        timesteps = torch.linspace(999, 0, n_steps, device=device).long()
        start_t = timesteps[n_steps - start_step]

        noise = torch.randn_like(inp_lat)
        latent = scheduler.add_noise(inp_lat, noise, start_t.unsqueeze(0))

        for idx in range(n_steps - start_step, n_steps):
            i = int(timesteps[idx])
            i_next = int(timesteps[idx+1]) if idx+1 < n_steps else 0
            t = torch.full((1,), i, device=device, dtype=torch.long)
            np_uncond = unet(latent, t, encoder_hidden_states=enc_uncond).sample
            np_cond   = unet(latent, t, encoder_hidden_states=enc_cond).sample
            noise_pred = np_uncond + guidance * (np_cond - np_uncond)
            a_t = scheduler.alpha_cumprod[i]
            a_next = scheduler.alpha_cumprod[i_next] if i_next > 0 else torch.tensor(1.0, device=device)
            pred_x0 = (latent - torch.sqrt(1-a_t)*noise_pred) / torch.sqrt(a_t)
            pred_x0 = pred_x0.clamp(-4, 4)
            latent = torch.sqrt(a_next)*pred_x0 + torch.sqrt(1-a_next)*noise_pred

        gen = vae.decode(latent / 0.18215).sample
        return (gen.clamp(-1, 1) + 1) / 2

gen = generate(inp_sd, inp_clip, rel_pose, GUIDANCE, STRENGTH)

def to_np(x):
    return x[0].cpu().permute(1,2,0).float().numpy().clip(0,1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_np((inp_sd+1)/2)); axes[0].set_title('Input Photo')
axes[1].imshow(to_np(gen)); axes[1].set_title(f'Generated @ rel_pose')
axes[2].imshow(to_np((gt+1)/2)); axes[2].set_title('Ground Truth Target')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/mv_single.png', dpi=150)
plt.show()

mse = ((gen - (gt+1)/2)**2).mean().item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
print(f'PSNR: {psnr:.2f} dB')

## 7. Generate 8 multi-view images (orbit around input)

In [ ]:
# Create 8 relative poses orbiting the input camera
def make_orbit_poses(n=8, radius=0.5, height=0.0):
    poses = []
    for i in range(n):
        theta = 2 * np.pi * i / n
        # Translation: orbit on horizontal circle
        tx = radius * np.sin(theta)
        tz = radius * (np.cos(theta) - 1)  # start at origin
        ty = height
        # Rotation: yaw to face the original origin
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], dtype=np.float32)
        T = np.eye(4, dtype=np.float32)
        T[:3, :3] = R
        T[:3, 3] = [tx, ty, tz]
        poses.append(T)
    return torch.from_numpy(np.stack(poses))  # (n, 4, 4)

orbit = make_orbit_poses(n=8, radius=0.3).to(device)

# Generate 8 views from same input
views = []
for i in range(8):
    pose_i = orbit[i:i+1]
    g = generate(inp_sd, inp_clip, pose_i, guidance=5.0, strength=0.7)
    views.append(to_np(g))
    print(f'  view {i+1}/8 done')

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0,0].imshow(to_np((inp_sd+1)/2)); axes[0,0].set_title('INPUT')
for i in range(8):
    r, c = (0, i+1) if i < 4 else (1, i-3)
    axes[r, c].imshow(views[i])
    axes[r, c].set_title(f'view {i+1}')
axes[1, 0].axis('off')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Multi-View Generation (orbit)')
plt.tight_layout()
plt.savefig('/content/mv_orbit.png', dpi=150)
plt.show()
print('Saved /content/mv_orbit.png')